# Assignment of wave 2 labs to new enumerators.

In [2]:
# Set date and flag for first instance
NEW_ENUMERATORS_DATE = "2026_06_12" # Date of new enumerators list, in YYYY_MM_DD format
FIRST_NEW_ENUMS_EVER = False # Set to False if new enumerators have already been given any labs assignment (wave 1 or wave 2)

In [3]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
import os
from lab_assignment import assign_enumerators

In [4]:
# Load datasets
labs = pd.read_csv(config.WAVE2_LABS_LIST / "LabsList_Randomized_Wave2.csv")
unassigned_labs = pd.read_csv(config.WAVE2_LABS_LIST / "LabsList_Unassigned_Wave2.csv")
wave2_existing_assignments = pd.read_csv(config.WAVE2_ENUMERATORS / "assignedlabs.csv")
combined_waves_existing_assignments = pd.read_csv(config.COMBINED_WAVES_ENUMERATORS / "assignedlabs_combined_waves.csv")
new_enumerators = pd.read_excel(config.WAVE2_ENUMERATORS / f"enumerators_new_{NEW_ENUMERATORS_DATE}.xlsx")
all_enumerators = pd.read_csv(config.COMBINED_WAVES_ENUMERATORS / "all_enumerators.csv")

In [5]:
# If the enumerators have ever been assigned labs, find them in list of all enumerators
if not FIRST_NEW_ENUMS_EVER:
    new_enumerators = new_enumerators.merge(all_enumerators, on=["enum_id"], how="left")

# Create necessary id variable
new_enumerators["id"] = new_enumerators["enum_id"]

In [6]:
# Combining data and defining no of treated and control labs to assign
np.random.seed(110)
enums_for_assignment = new_enumerators.copy()

# Create n_treat and n_control columns if don't already exist
for col in ["n_treated", "n_control"]:
    if col not in enums_for_assignment:
        enums_for_assignment[col] = pd.NA

# Replace with extra_total/2
def fill_treat_control(row):

    extra_total = row.get("extra_total", pd.NA)

    if extra_total % 2 == 0:  # deal with even numbers
        half = extra_total // 2
        return half, half
    else:  # deal with odd numbers
            base = extra_total // 2
            if np.random.rand() < 0.5:
                return base, base + 1
            else:
                return base + 1, base
    if pd.notna(extra_total):
        half = extra_total // 2
        return half, extra_total - half  # deal with odd numbers

    return n_t, n_c

enums_for_assignment[["n_treated", "n_control"]] = (
    enums_for_assignment.apply(fill_treat_control, axis=1, result_type="expand")
)

In [7]:
# Run assignment
assignments, leftover_treatment, leftover_control = assign_enumerators(
    labs_df = unassigned_labs,
    enum_df = enums_for_assignment,
    n_treat = 3,
    n_control = 3,
    seed = 110
)

# Check for duplicate assignments
duplicates = assignments[assignments.duplicated(subset="labgroupid", keep=False)]
if not duplicates.empty:
    print("Duplicate labgroupids found:")
    print(duplicates[["labgroupid", "Lab Group", "enum_firstname", "enum_lastname"]])
else:
    print("No duplicates of labgroupid found.")

# Order assignments by enumerator id and labgroupid
assignments = assignments.sort_values(by=["enum_id", "labgroupid"]).reset_index(drop=True)

# Add a new column if out of sample doesn’t already exist
if "out_of_sample" not in wave2_existing_assignments.columns:
    wave2_existing_assignments["out_of_sample"] = 0

# Add a new column if it doesn’t already exist
if "out_of_sample" not in assignments.columns:
    assignments["out_of_sample"] = 0

No duplicates of labgroupid found.


In [8]:
# Combine with existing assignments and check for duplicates

# wave 2 only
wave2_assignments = pd.concat([wave2_existing_assignments, assignments]).reset_index(drop=True)

# Check for duplicate assignments
duplicates = wave2_assignments[wave2_assignments.duplicated(subset="labgroupid", keep=False)]
if not duplicates.empty:
    print("Duplicate labgroupids found in wave 2 assignments:")
    print(duplicates[["labgroupid", "Lab Group", "enum_firstname", "enum_lastname"]])
else:
    print("No duplicates of labgroupid found in wave 2 assignments.")

# combined waves
all_assignments = pd.concat([combined_waves_existing_assignments, assignments]).reset_index(drop=True)

# Check for duplicate assignments
duplicates = all_assignments[all_assignments.duplicated(subset="labgroupid", keep=False)]
if not duplicates.empty:
    print("Duplicate labgroupids found in all assignments:")
    print(duplicates[["labgroupid", "Lab Group", "enum_firstname", "enum_lastname"]])
else:
    print("No duplicates of labgroupid found in all assignments.")

No duplicates of labgroupid found in wave 2 assignments.
No duplicates of labgroupid found in all assignments.


In [9]:
# Save the files

# Reorder columns for saving assignments file
assignments_order = [
    "labgroupid", "Lab Group", "Faculty", "Institute", 
    "Professor", "Email", "Source", "Treatment Status", 
    "enum_id", "enum_firstname", "enum_lastname", 
    "enum_email", "out_of_sample"
]

# Save the assignments files
cols_to_save = [col for col in assignments_order if col in assignments.columns]
assignments.to_csv(config.WAVE2_ENUMERATORS / f"extra_assignedlabs_{NEW_ENUMERATORS_DATE}.csv", index=False, columns=cols_to_save)
wave2_assignments.to_csv(config.WAVE2_ENUMERATORS / f"assignedlabs.csv", index=False, columns=cols_to_save)
all_assignments.to_csv(config.COMBINED_WAVES_ENUMERATORS / "assignedlabs_combined_waves.csv", index = False, columns = cols_to_save)

In [10]:
# Leftover labs

# Combine leftover labs
unassigned_labs = pd.concat([leftover_treatment, leftover_control]).reset_index(drop=True)

# Check that no labgroupid is in both assignments and leftover labs
assigned_ids = set(assignments["labgroupid"])
unassigned_ids = set(unassigned_labs["labgroupid"])
overlap = assigned_ids & unassigned_ids
if overlap:
    print("Error: Some labgroupids are in both assignments and unassigned_labs:")
    print(overlap)
else:
    print("All labgroupids are correctly assigned or unassigned.")

# Save unassigned labs (if it is not empty)
unassigned_labs.to_csv(config.WAVE2_LABS_LIST / "LabsList_Unassigned_Wave2.csv", index=False)
if unassigned_labs.empty:
    print("No unassigned labs left.")

All labgroupids are correctly assigned or unassigned.
No unassigned labs left.


In [11]:
# Create assignments file for each enumerator

# Rename columns for clarity
assignments = assignments.rename(columns={"Source": "Website"})

# Columns to include in the enumerator's file
cols_to_include = [
    "labgroupid", "Lab Group", "Faculty", "Institute", 
    "Professor", "Email", "Website", "Treatment Status",
    "Contact person (if different to prof)", "Contact email (if different)",
    "Baseline visit status", "Endline visit status", "Comments"
]

# Color treatment yellow and control no color
fill_colors = {
    "treatment": "FFFF00",  # Yellow
    "control": "FFFFFF"     # No color (white)
}

for enum_id, enum_data in assignments.groupby("enum_id"):

    # Get enumerator info
    id = enum_data["enum_id"].iloc[0]
    name = enum_data["enum_foldername"].iloc[0]
    folder_name = f"{name}_data"

    # Columns to include — reindex adds missing cols as NaN rather than raising KeyError
    labs_for_enum = enum_data.reindex(columns=cols_to_include)

    # Create excel path
    filename = os.path.join(config.SWITCHDRIVE_ROOT, folder_name, f"lab_assignment_{NEW_ENUMERATORS_DATE}.xlsx")

    # Save first without formatting
    labs_for_enum.to_excel(filename, index=False)

    # Load workbook and select active sheet
    wb = load_workbook(filename)
    ws = wb.active
    ws.title = "Lab Assignments"

    # Bold header row
    for cell in ws[1]:
        cell.font = Font(bold=True)

    # Adjust column widths
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter  # Get the column name
        for cell in col:
            try:
                if cell.value:
                    max_length = max(max_length, len(str(cell.value)))
            except:
                pass
        adjusted_width = (max_length + 2)
        ws.column_dimensions[column].width = adjusted_width

    # Fill colors based on treatment status
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=ws.min_column, max_col=ws.max_column):
        status_cell = row[cols_to_include.index("Treatment Status")]
        status = str(status_cell.value).lower()
        if status in fill_colors:
            status_cell.fill = PatternFill(start_color=fill_colors[status], end_color=fill_colors[status], fill_type="solid")
    
    # Save workbook
    wb.save(filename)